# 1. Bibliotecas Utilizadas

In [23]:
import os
import sys
import pandas as pd
import warnings
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType

---
# 2. Configurações Globais & Funções Utilizadas

## 2.1 Configurações

In [2]:
# Garante o apontamento direto para a JVM instalada
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
# Retirar truncamento de colunas e registros
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [4]:
# Evitar de mostrar Warnings
warnings.filterwarnings("ignore")

## 2.2 Funções

In [15]:
# Verificação de valores ausentes
def inspect_missing(df, columns):
    missing_exprs = []
    for c in columns:
        col_type = df.schema[c].dataType.simpleString()
        if col_type in ("double", "float"):
            expr = F.count(F.when(F.col(c).isNull() | F.isnan(F.col(c)), c)).alias(c)
        else:
            expr = F.count(F.when(F.col(c).isNull(), c)).alias(c)
        missing_exprs.append(expr)
    
    total_rows = df.count()
    df_missing = df.select(missing_exprs).toPandas().T.reset_index()
    df_missing.columns = ["coluna", "nulos_qtd"]
    df_missing["nulos_pct"] = (df_missing["nulos_qtd"] / total_rows) * 100
    return df_missing.sort_values(by="nulos_pct", ascending=False)

---
# 3. Inicializando o Spark e Carregando os Dados 

In [5]:
# Inicialização com alocação balanceada para 32GB RAM e múltiplos núcleos
spark = (
    SparkSession.builder.appName("HomeCredit-Medallion-Architecture")
    .master("local[*]")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.maxResultSize", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/28 14:10:26 WARN Utils: Your hostname, jean-Legion-Y540-15IRH, resolves to a loopback address: 127.0.1.1; using 192.168.15.7 instead (on interface enp8s0)
26/10/28 14:10:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 14:10:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.2.1


Para este ciclo de desenvolvimento (MVP do motor preditivo), o escopo de ingestão da camada Bronze foi delimitado às tabelas `application_train (dados cadastrais e proposta)` e `bureau (histórico de crédito externo)`. As demais fontes transacionais de histórico detalhado de parcelas e cartões foram postergadas para iterações futuras do produto, priorizando a estabilidade do contrato de inferência e a agilidade na esteira de deploy.

In [6]:
# Ingestão a partir dos caminhos relativos na pasta data/
df_app_bronze = spark.read.csv(
    "../data/bronze/application_train.csv",
    header=True,
    inferSchema=True
)

df_bureau_bronze = spark.read.csv(
    "../data/bronze/bureau.csv",
    header=True,
    inferSchema=True
)

print(f"Bronze application_train: {df_app_bronze.count()} linhas, {len(df_app_bronze.columns)} colunas")
print(f"Bronze bureau: {df_bureau_bronze.count()} linhas, {len(df_bureau_bronze.columns)} colunas")

Bronze application_train: 307511 linhas, 122 colunas
Bronze bureau: 1716428 linhas, 17 colunas


---
# 4. Camada Bronze $\rightarrow$ Silver: Inspeção e Tratamento Inicial

## 4.1 Inspeção inicial de Tipagem e Anomalias

In [7]:
df_app_bronze.limit(10).toPandas()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [8]:
df_bureau_bronze.limit(10).toPandas()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.00,0.00,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.00,171342.00,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.50,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.00,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.00,NaN,NaN,0.0,Consumer credit,-21,NaN
5,215354,5714467,Active,currency 1,-273,0,27460.0,NaN,0.0,0,180000.00,71017.38,108982.62,0.0,Credit card,-31,NaN
6,215354,5714468,Active,currency 1,-43,0,79.0,NaN,0.0,0,42103.80,42103.80,0.00,0.0,Consumer credit,-22,NaN
7,162297,5714469,Closed,currency 1,-1896,0,-1684.0,-1710.0,14985.0,0,76878.45,0.00,0.00,0.0,Consumer credit,-1710,NaN
8,162297,5714470,Closed,currency 1,-1146,0,-811.0,-840.0,0.0,0,103007.70,0.00,0.00,0.0,Consumer credit,-840,NaN
9,162297,5714471,Active,currency 1,-1146,0,-484.0,NaN,0.0,0,4500.00,0.00,0.00,0.0,Credit card,-690,NaN


#### Critério de Seleção Preliminar de Variáveis para Inspeção (Metodologia dos 5 Cs do Crédito)

A tabela bruta `application_train` possui mais de 120 colunas. Para viabilizar a análise exploratória inicial e evitar o processamento de atributos com baixo sinal preditivo ou alto ruído, a triagem das variáveis prioritárias para inspeção baseia-se na metodologia clássica dos **5 Cs do Crédito** (*Capacidade, Caráter, Colateral, Capital e Condições*):

* **Capacidade & Condições (Renda e Compromissos):** `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_ANNUITY` e `AMT_GOODS_PRICE`. Mensuram a capacidade financeira do proponente e a relação entre o valor da parcela e a renda declarada.
* **Caráter (Perfil e Estabilidade):** `DAYS_BIRTH`, `DAYS_EMPLOYED`, `NAME_INCOME_TYPE` e `NAME_EDUCATION_TYPE`. Avaliam a maturidade, estabilidade no emprego e o contexto socioeconômico do tomador.
* **Colateral & Capital (Patrimônio):** `FLAG_OWN_CAR` e `FLAG_OWN_REALTY`. Indicam a posse de bens duráveis que servem como indicativo de reserva patrimonial ou potencial garantia.
* **Histórico Externo (Mercado):** `EXT_SOURCE_2`, `EXT_SOURCE_3` e os registros agregados da base `bureau`. Avaliam o comportamento de crédito preexistente e a alavancagem do cliente perante outras instituições financeiras.

#### Motivação para Não Priorizar as Demais Variáveis nesta Etapa:
* **Características Habitacionais (`APARTMENTS_*`, `BASEMENTAREA_*`, etc.):** Descrições físicas prediais que tipicamente possuem alto volume de dados ausentes e pouca relação direta com a solvência financeira imediata.
* **Flags Administrativas de Documentação (`FLAG_DOCUMENT_2` a `21`):** Controles burocráticos internos que, em sua grande maioria, apresentam variância quase nula, sem relevância discriminatória preliminar.

In [ ]:
# DEFINIÇÃO DAS VARIÁVEIS TRIADAS (5 Cs DO CRÉDITO)
cols_app = [
    "SK_ID_CURR", "TARGET", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
    "AMT_GOODS_PRICE", "DAYS_BIRTH", "DAYS_EMPLOYED", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "EXT_SOURCE_2", "EXT_SOURCE_3"
]

cols_bureau = [
    "SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "DAYS_CREDIT",
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT"
]

df_app_filtered = df_app_bronze.select(cols_app)
df_bureau_filtered = df_bureau_bronze.select(cols_bureau)

In [11]:
# 1. ESTATÍSTICAS DESCRITIVAS COM PERCENTIS (APPLICATION_TRAIN)
numeric_app = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "EXT_SOURCE_2", "EXT_SOURCE_3"
]

print("--- SUMÁRIO NUMÉRICO (APPLICATION_TRAIN) ---")
df_app_filtered.select(numeric_app).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "99%", "max"
).show(truncate=False)

# 2. ESTATÍSTICAS DESCRITIVAS COM PERCENTIS (BUREAU)
numeric_bureau = ["DAYS_CREDIT", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT"]

print("--- SUMÁRIO NUMÉRICO (BUREAU) ---")
df_bureau_filtered.select(numeric_bureau).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "99%", "max"
).show(truncate=False)

--- SUMÁRIO NUMÉRICO (APPLICATION_TRAIN) ---


+-------+-----------------+-----------------+------------------+-----------------+-------------------+------------------+--------------------+--------------------+
|summary|AMT_INCOME_TOTAL |AMT_CREDIT       |AMT_ANNUITY       |AMT_GOODS_PRICE  |DAYS_BIRTH         |DAYS_EMPLOYED     |EXT_SOURCE_2        |EXT_SOURCE_3        |
+-------+-----------------+-----------------+------------------+-----------------+-------------------+------------------+--------------------+--------------------+
|count  |307511           |307511           |307499            |307233           |307511             |307511            |306851              |246546              |
|mean   |168797.9192969845|599025.9997057016|27108.573909183444|538396.2074288895|-16036.995066843137|63815.04590404896 |0.5143926741308462  |0.5108529061799736  |
|stddev |237123.146278856 |402490.7769958547|14493.737315118331|369446.4605400569|4363.988631785563  |141275.76651872732|0.19106015498493661 |0.1948443644637489  |
|min    |25650.0

[Stage 23:==========================================>              (9 + 3) / 12]

+-------+------------------+------------------+-------------------+
|summary|DAYS_CREDIT       |AMT_CREDIT_SUM    |AMT_CREDIT_SUM_DEBT|
+-------+------------------+------------------+-------------------+
|count  |1716428           |1716415           |1458759            |
|mean   |-1142.107684680045|354994.59191767214|137085.11995216066 |
|stddev |795.1649282053268 |1149811.3439797729|677401.1309521444  |
|min    |-2922             |0.0               |-4705600.32        |
|25%    |-1666             |51300.0           |0.0                |
|50%    |-987              |125509.5          |0.0                |
|75%    |-474              |315000.0          |40126.5            |
|99%    |-38               |3687300.0         |2252286.0          |
|max    |0                 |5.85E8            |1.701E8            |
+-------+------------------+------------------+-------------------+



In [ ]:
                                                                            
'''
--- SUMÁRIO NUMÉRICO (APPLICATION_TRAIN) ---
+-------+-----------------+-----------------+------------------+-----------------+-------------------+------------------+--------------------+--------------------+
|summary|AMT_INCOME_TOTAL |AMT_CREDIT       |AMT_ANNUITY       |AMT_GOODS_PRICE  |DAYS_BIRTH         |DAYS_EMPLOYED     |EXT_SOURCE_2        |EXT_SOURCE_3        |
+-------+-----------------+-----------------+------------------+-----------------+-------------------+------------------+--------------------+--------------------+
|count  |307511           |307511           |307499            |307233           |307511             |307511            |306851              |246546              |
|mean   |168797.9192969845|599025.9997057016|27108.573909183444|538396.2074288895|-16036.995066843137|63815.04590404896 |0.5143926741308462  |0.5108529061799736  |
|stddev |237123.146278856 |402490.7769958547|14493.737315118331|369446.4605400569|4363.988631785563  |141275.76651872732|0.19106015498493661 |0.1948443644637489  |
|min    |25650.0          |45000.0          |1615.5            |40500.0          |-25229             |-17912            |8.173616518884397E-8|5.272652387098817E-4|
|25%    |112500.0         |270000.0         |16524.0           |238500.0         |-19682             |-2761             |0.392420918862483   |0.3706496323299817  |
|50%    |146250.0         |513531.0         |24903.0           |450000.0         |-15751             |-1214             |0.5659291952716554  |0.5352762504724826  |
|75%    |202500.0         |808650.0         |34596.0           |679500.0         |-12415             |-290              |0.6635926818710476  |0.6690566947824041  |
|99%    |468000.0         |1845000.0        |69912.0           |1800000.0        |-8266              |365243            |0.7826886949727564  |0.8327850252992314  |
|max    |1.17E8           |4050000.0        |258025.5          |4050000.0        |-7489              |365243            |0.8549996664047012  |0.8960095494948396  |
+-------+-----------------+-----------------+------------------+-----------------+-------------------+------------------+--------------------+--------------------+

--- SUMÁRIO NUMÉRICO (BUREAU) ---

[Stage 23:==========================================>              (9 + 3) / 12]

+-------+------------------+------------------+-------------------+
|summary|DAYS_CREDIT       |AMT_CREDIT_SUM    |AMT_CREDIT_SUM_DEBT|
+-------+------------------+------------------+-------------------+
|count  |1716428           |1716415           |1458759            |
|mean   |-1142.107684680045|354994.59191767214|137085.11995216066 |
|stddev |795.1649282053268 |1149811.3439797729|677401.1309521444  |
|min    |-2922             |0.0               |-4705600.32        |
|25%    |-1666             |51300.0           |0.0                |
|50%    |-987              |125509.5          |0.0                |
|75%    |-474              |315000.0          |40126.5            |
|99%    |-38               |3687300.0         |2252286.0          |
|max    |0                 |5.85E8            |1.701E8            |
+-------+------------------+------------------+-------------------+

'''

#### Diagnóstico Técnico e Racional das Transformações (Bronze $\rightarrow$ Silver)

A inspeção estatística descritiva das tabelas confirma padrões estruturais de modelagem temporal, a presença de valores sentinela e distribuições com assimetria severa:

##### 1. Tabela `application_train`

* **Anomalia Crítica em `DAYS_EMPLOYED`:** A média é artificialmente positiva ($+63.815$ dias) devido ao valor extremo de **`365243`** (presente no percentil de 99% e no valor máximo). Este valor sentinela representa ausência de vínculo formal ou aposentadoria e deve ser convertido para `Null` na camada Silver, mantendo a integridade matemática da distribuição.
* **Modelagem Temporal Relativa (`DAYS_BIRTH`):** Valores estritamente negativos ancorados na data da proposta (mínimo de $-25.229$ e máximo de $-7.489$, correspondendo à faixa etária de **20,5 a 69 anos**). A transformação para escala positiva em anos (`idade = DAYS_BIRTH / -365.25`) é necessária para a interpretabilidade de negócio e consumo na API.
* **Assimetria em Métricas Financeiras:** 
  * `AMT_INCOME_TOTAL`: A mediana situa-se em $146.250$, contudo o valor máximo atinge $1.17 \times 10^8$ (117 milhões), evidenciando uma forte assimetria à direita (*outliers* extremos). Isso reforça a necessidade de pré-processamento robusto (imputação por mediana e escalonamento) no pipeline de ML.
  * `AMT_CREDIT`, `AMT_ANNUITY` e `AMT_GOODS_PRICE`: Apresentam distribuições coerentes com o financiamento de consumo (média de crédito em torno de $599.000$), com baixa volumetria de nulos.
* **Escores de Bureau (`EXT_SOURCE_2` e `EXT_SOURCE_3`):** Ambas as variáveis estão normalizadas entre $0$ e $1$, com médias próximas de $0.51$. Observa-se que `EXT_SOURCE_3` possui ausência em aproximadamente **$19,8\%$ da base** ($246.546$ registos preenchidos de $307.511$), exigindo estratégia de imputação no modelo.

##### 2. Tabela `bureau`

* **Janela Temporal (`DAYS_CREDIT`):** Variação de $-2.922$ a $0$ dias, demonstrando um histórico consolidado de até **8 anos** anteriores à proposta atual.
* **Endividamento e Inconsistência Contábil (`AMT_CREDIT_SUM_DEBT`):**
  * A mediana é $0.0$, demonstrando que mais de 50% dos contratos registados já foram liquidados ou estão sem saldo devedor.
  * **Anomalia Detetada:** Presença de valor mínimo negativo (**$-4.705.600,32$**). Valores de dívida negativos representam créditos excedentes ou estornos no bureau, ou erro de inserção bancária; para fins de risco, devem ser tratados (ex.: limitados a zero via `F.greatest(col, 0)`) para evitar distorções no cálculo do endividamento real.

In [14]:
# Inspeção do contrato de tipos gerado pelo Spark na ingestão
print("--- Schema Application Train ---")
df_app_bronze.select(cols_app ).printSchema()

print("--- Schema Bureau ---")
df_bureau_bronze.select(cols_bureau).printSchema()

--- Schema Application Train ---
root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- EXT_SOURCE_2: double (nullable = true)
 |-- EXT_SOURCE_3: double (nullable = true)

--- Schema Bureau ---
root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- CREDIT_ACTIVE: string (nullable = true)
 |-- DAYS_CREDIT: integer (nullable = true)
 |-- AMT_CREDIT_SUM: double (nullable = true)
 |-- AMT_CREDIT_SUM_DEBT: double (nullable = true)



#### Avaliação Estrutural de Esquema e Tipagem (Bronze $\rightarrow$ Silver)

A inspeção do esquema inferido pelo PySpark (`inferSchema=True`) valida que as variáveis triadas foram mapeadas com precisão física, mas exige coerção semântica e padronização antes da persistência na camada Silver:

##### 1. Tabela `application_train`
* **Identificador (`SK_ID_CURR`) e Target (`TARGET`):** Mapeados corretamente como `IntegerType`. O identificador será mantido exclusivamente como chave de rastreamento e junção, enquanto `TARGET` preserva a sua integridade binária ($0$ e $1$).
* **Métricas Financeiras e Escores (`DoubleType`):** `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE`, `EXT_SOURCE_2` e `EXT_SOURCE_3` vieram como ponto flutuante de dupla precisão, assegurando a exatidão necessária para operações aritméticas monetárias e probabilidades contínuas.
* **Variáveis Temporais (`IntegerType`):** `DAYS_BIRTH` e `DAYS_EMPLOYED` foram lidas como inteiros. Na camada Silver, serão convertidas para escala positiva e anualizada (`DoubleType`), eliminando a anomalia sentinela de emprego e os valores negativos relativos.
* **Flags Patrimoniais (`StringType`):** `FLAG_OWN_CAR` e `FLAG_OWN_REALTY` apresentam-se como texto (`'Y'` e `'N'`). Devem ser convertidas em inteiros binários ($1$ e $0$) na Silver para padronização analítica e menor pegada de memória.
* **Atributos Categóricos (`StringType`):** `NAME_INCOME_TYPE` e `NAME_EDUCATION_TYPE` mantêm-se como string para tratamento posterior via codificação categórica (*One-Hot Encoding*) no pipeline de modelagem.

##### 2. Tabela `bureau`
* **Chaves Relacionais (`SK_ID_CURR`, `SK_ID_BUREAU`):** Tipadas como `IntegerType`, adequadas para a relação $1:N$ e agregação por proponente na transição para a camada Gold.
* **Status do Contrato (`CREDIT_ACTIVE`):** Preserva o tipo `StringType`, permitindo a segregação lógica de contratos ativos versus encerrados nas métricas agregadas de endividamento.
* **Métricas Temporais e de Saldo (`IntegerType` / `DoubleType`):** `DAYS_CREDIT` (tempo relativo em dias) e os saldos `AMT_CREDIT_SUM` e `AMT_CREDIT_SUM_DEBT` foram tipados corretamente, viabilizando operações diretas de soma, média e identificação de valores máximos por cliente.

In [17]:
# AUDITORIA DE NULOS E NaNs

print("--- VALORES AUSENTES: APPLICATION_TRAIN ---")
print(inspect_missing(df_app_filtered, cols_app))
print()
print("\n--- VALORES AUSENTES: BUREAU ---")
print(inspect_missing(df_bureau_filtered, cols_bureau))

--- VALORES AUSENTES: APPLICATION_TRAIN ---
                 coluna  nulos_qtd  nulos_pct
13         EXT_SOURCE_3      60965  19.825307
12         EXT_SOURCE_2        660   0.214626
5       AMT_GOODS_PRICE        278   0.090403
4           AMT_ANNUITY         12   0.003902
1                TARGET          0   0.000000
0            SK_ID_CURR          0   0.000000
2      AMT_INCOME_TOTAL          0   0.000000
3            AMT_CREDIT          0   0.000000
6            DAYS_BIRTH          0   0.000000
7         DAYS_EMPLOYED          0   0.000000
9   NAME_EDUCATION_TYPE          0   0.000000
8      NAME_INCOME_TYPE          0   0.000000
11      FLAG_OWN_REALTY          0   0.000000
10         FLAG_OWN_CAR          0   0.000000


--- VALORES AUSENTES: BUREAU ---
                coluna  nulos_qtd  nulos_pct
5  AMT_CREDIT_SUM_DEBT     257669  15.011932
4       AMT_CREDIT_SUM         13   0.000757
1         SK_ID_BUREAU          0   0.000000
0           SK_ID_CURR          0   0.000000
3     

#### Diagnóstico de Valores Ausentes e Estratégia de Imputação (Bronze $\rightarrow$ Silver)

A análise da volumetria de valores nulos valida a completude das variáveis selecionadas e fundamenta a necessidade de imputação estatística posterior via Scikit-Learn (durante o pipeline de treinamento), preservando a camada Silver isenta de vazamento de dados (data leakage).

##### 1. Tabela `application_train`
* **Volumetria Relevante em `EXT_SOURCE_3`:** Apresenta **$19,83\%$ de ausência** ($60.965$ registros). Como os escores externos constituem preditores de alto impacto para o risco de crédito, a variável deve ser preservada. A imputação (por exemplo, via mediana) será delegada ao `ColumnTransformer` do Scikit-Learn no pipeline offline para evitar vazamento de dados (*data leakage*).
* **Ausências Residuais (< 1%):**
  * `EXT_SOURCE_2`: Apresenta apenas **$0,21\%$ de nulos** ($660$ registros).
  * `AMT_GOODS_PRICE` e `AMT_ANNUITY`: Registram **$0,09\%$** ($278$ casos) e **$0,004\%$** ($12$ casos), respectivamente. Por representarem frações mínimas, a imputação por mediana no pré-processamento resolverá a integridade sem distorcer as distribuições originais[cite: 3].
* **Completude Estrutural ($0\%$ de Nulos):** Variáveis essenciais como identificador (`SK_ID_CURR`), variável-alvo (`TARGET`), métricas financeiras (`AMT_INCOME_TOTAL`, `AMT_CREDIT`), datas relativas (`DAYS_BIRTH`, `DAYS_EMPLOYED`) e atributos categóricos/flags (`NAME_*`, `FLAG_*`) estão $100\%$ preenchidas, garantindo solidez para a modelagem.

##### 2. Tabela `bureau`
* **Saldo Devedor (`AMT_CREDIT_SUM_DEBT`):** Registra **$15,01\%$ de ausência** ($257.669$ linhas). No contexto de crédito, valores nulos nesse campo frequentemente indicam contratos já liquidados, linhas de crédito não utilizadas ou ausência de saldo residual. Na transição para a camada Gold, essa coluna deve ser tratada (preenchimento com `0.0`) antes da agregação da dívida total por cliente.
* **Completude das Demais Métricas:** O montante total do crédito (`AMT_CREDIT_SUM`) possui apenas $13$ nulos ($0,0007\%$), enquanto as chaves de junção (`SK_ID_CURR`, `SK_ID_BUREAU`), o status (`CREDIT_ACTIVE`) e a data relativa (`DAYS_CREDIT`) possuem $0\%$ de dados ausentes, assegurando agregações consistentes sem perda de clientes na junção relacional.

### 4.1.1 Evidenciação das Anomalias de Negócio

In [18]:
# INVESTIGAÇÃO 1: O VALOR SENTINELA 365243 (DAYS_EMPLOYED)
print("--- CRUZAMENTO: DAYS_EMPLOYED == 365243 vs NAME_INCOME_TYPE ---")
(
    df_app_filtered
    .groupBy("NAME_INCOME_TYPE")
    .agg(
        F.count("*").alias("total_registros"),
        F.sum(F.when(F.col("DAYS_EMPLOYED") == 365243, 1).otherwise(0)).alias("qtd_valor_sentinela"),
        F.min("DAYS_EMPLOYED").alias("min_days_employed"),
        F.max("DAYS_EMPLOYED").alias("max_days_employed")
    )
    .withColumn("pct_sentinela", (F.col("qtd_valor_sentinela") / F.col("total_registros")) * 100)
    .orderBy(F.desc("qtd_valor_sentinela"))
    .show(truncate=False)
)

--- CRUZAMENTO: DAYS_EMPLOYED == 365243 vs NAME_INCOME_TYPE ---
+--------------------+---------------+-------------------+-----------------+-----------------+-----------------+
|NAME_INCOME_TYPE    |total_registros|qtd_valor_sentinela|min_days_employed|max_days_employed|pct_sentinela    |
+--------------------+---------------+-------------------+-----------------+-----------------+-----------------+
|Pensioner           |55362          |55352              |-12865           |365243           |99.98193706874751|
|Unemployed          |22             |22                 |365243           |365243           |100.0            |
|Commercial associate|71617          |0                  |-17546           |-2               |0.0              |
|Student             |18             |0                  |-8005            |-191             |0.0              |
|State servant       |21703          |0                  |-17583           |-1               |0.0              |
|Working             |158774    

In [19]:
# INVESTIGAÇÃO 2: VALORES TEMPORAIS NEGATIVOS (DAYS_BIRTH E DAYS_CREDIT)
print("--- INTERVALO TEMPORAL DE DIAS ---")
df_app_filtered.select(
    F.min("DAYS_BIRTH").alias("min_days_birth"),
    F.max("DAYS_BIRTH").alias("max_days_birth")
).show()

df_bureau_filtered.select(
    F.min("DAYS_CREDIT").alias("min_days_credit"),
    F.max("DAYS_CREDIT").alias("max_days_credit")
).show()

--- INTERVALO TEMPORAL DE DIAS ---
+--------------+--------------+
|min_days_birth|max_days_birth|
+--------------+--------------+
|        -25229|         -7489|
+--------------+--------------+

+---------------+---------------+
|min_days_credit|max_days_credit|
+---------------+---------------+
|          -2922|              0|
+---------------+---------------+



In [20]:
# INVESTIGAÇÃO 3: AUDITORIA DE STATUS E VALORES EM BUREAU

print("--- DISTRIBUIÇÃO DE CREDIT_ACTIVE (BUREAU) ---")
df_bureau_filtered.groupBy("CREDIT_ACTIVE").count().orderBy(F.desc("count")).show()

--- DISTRIBUIÇÃO DE CREDIT_ACTIVE (BUREAU) ---
+-------------+-------+
|CREDIT_ACTIVE|  count|
+-------------+-------+
|       Closed|1079273|
|       Active| 630607|
|         Sold|   6527|
|     Bad debt|     21|
+-------------+-------+



#### Diagnóstico de Anomalias Estruturais e Distribuições Categóricas (Bronze $\rightarrow$ Silver)

A inspeção empírica cruzada valida a natureza das anomalias temporais e delimita o comportamento dos registros para a higienização da camada Silver:

##### 1. Confirmação do Valor Sentinela (`DAYS_EMPLOYED == 365243`)
* **Padrão Ocupacional:** A ocorrência de `365243` concentra-se integralmente em **`Pensioner`** ($99,98\%$, representando $55.352$ casos) e **`Unemployed`** ($100\%$, totalizando $22$ casos).
* **Ausência nos Demais Vínculos:** Categorias com atividade laboral ativa (`Working`, `Commercial associate`, `State servant`, etc.) apresentam $0\%$ do valor anômalo, possuindo contagens estritamente negativas (máximos entre $-1$ e $0$ dias).
* **Decisão Técnica de Engenharia:** Fica comprovado que `365243` é um código sentinela de sistema legado para representar a ausência de vínculo empregatício formal ativo. Na transformação para a camada Silver, esse valor será convertido para `Null` (e os demais valores negativos convertidos para anos positivos), garantindo a fidelidade estatística da variável.

##### 2. Validação da Escala Temporal Reversa
* **Idade (`DAYS_BIRTH`):** Intervalo entre $-25.229$ e $-7.489$ dias (equivalente a uma amplitude real de **$20,5$ a $69,1$ anos**). Confirma que o marco zero ($t_0$) é a data da solicitação do crédito.
* **Histórico do Bureau (`DAYS_CREDIT`):** Variação de $-2.922$ a $0$ dias, abrangendo contratos abertos nos últimos **$8$ anos** até a data atual da consulta.

##### 3. Distribuição do Status de Contratos (`CREDIT_ACTIVE`)
* **Predomínio de Contratos Encerrados:** **$62,88\%$** ($1.079.273$ contratos) estão como `Closed`, enquanto **$36,74\%$** ($630.607$) permanecem como `Active`.
* **Casos Críticos Residuais:** Registros categorizados como `Sold` ($6.527$) e `Bad debt` ($21$) representam inadimplência severa ou cessão de crédito a terceiros, servindo como marcadores de alto risco nas agregações da camada Gold.

### 4.1.2 Resumo da inspeção

#### Síntese do Data Profiling e Especificação Técnica (Bronze $\rightarrow$ Silver)

A inspeção empírica realizada nas tabelas `application_train` e `bureau` na camada Bronze consolidou as evidências estatísticas e de negócio necessárias para a higienização dos dados:

1. **Higienização de Valores Sentinela:** O valor `365243` em `DAYS_EMPLOYED` concentra-se em proponentes das categorias `Pensioner` ($99,98\%$) e `Unemployed` ($100\%$). Representa a codificação legada para ausência de vínculo formal ativo e deve ser convertido estritamente para `Null`.
2. **Normalização Temporal:** As variáveis `DAYS_BIRTH` e `DAYS_EMPLOYED` seguem contagem negativa em dias retroativos a partir da data da proposta ($t_0$). Serão convertidas em anos positivos (`abs(DAYS) / 365.25`) para interpretabilidade analítica e compatibilidade com a futura API.
3. **Correção de Inconsistência Contábil:** O atributo `AMT_CREDIT_SUM_DEBT` (tabela `bureau`) apresenta valor mínimo anômalo negativo ($-4.705.600,32$). Valores inferiores a zero serão truncados para `0.0`.
4. **Governança de Dados Ausentes e Preservação da Silver:**
   - Valores nulos em scores externos (`EXT_SOURCE_3` com $19,83\%$, `EXT_SOURCE_2` com $0,21\%$) e métricas financeiras residuais serão **preservados como nulos** na camada Silver, evitando qualquer risco de vazamento de dados (*data leakage*). A imputação estatística será realizada exclusivamente no pipeline Scikit-Learn (Fase 1/2) após o particionamento de treino/teste.
   - Nulos de saldo devedor em contratos fechados do bureau serão tratados deterministicamente via agregação na camada Gold.
5. **Tipagem Estrita e Persistência:** Conversão de flags binárias (`FLAG_OWN_CAR`, `FLAG_OWN_REALTY`) de texto (`'Y'`/`'N'`) para inteiros (`1`/`0`), coerção explícita de tipos numéricos e gravação em disco no formato colunar `.parquet` com caminhos relativos em `data/silver/`.

---

#### Próximos Passos
1. Executar a transformação das tabelas `application_train` e `bureau` via PySpark aplicando as regras documentadas.
2. Persistir os DataFrames resultantes em `data/silver/application_train.parquet` e `data/silver/bureau.parquet`.
3. Validar a leitura e o esquema dos arquivos Parquet gerados na camada Silver.

# 4.2 Transformações e armazenamento na camada Silver

In [26]:
# Definição dos caminhos relativos da arquitetura medalhão
BRONZE_PATH = "../data/bronze"
SILVER_PATH = "../data/silver"

### 4.2.1 Transformações

In [27]:
# Transformação e Higienização: application_train

df_app_silver = df_app_bronze.select(
    # Chave e Target
    F.col("SK_ID_CURR").cast(IntegerType()),
    F.col("TARGET").cast(IntegerType()),

    # Métricas Financeiras (Capacidade e Condições)
    F.col("AMT_INCOME_TOTAL").cast(DoubleType()),
    F.col("AMT_CREDIT").cast(DoubleType()),
    F.col("AMT_ANNUITY").cast(DoubleType()),
    F.col("AMT_GOODS_PRICE").cast(DoubleType()),

    # Variáveis Temporais Normalizadas (Dias Negativos -> Anos Positivos)
    (F.abs(F.col("DAYS_BIRTH")) / 365.25).cast(DoubleType()).alias("AGE_YEARS"),
    
    # Tratamento da anomalia sentinela 365243
    F.when(F.col("DAYS_EMPLOYED") == 365243, None)
     .otherwise(F.abs(F.col("DAYS_EMPLOYED")) / 365.25)
     .cast(DoubleType())
     .alias("YEARS_EMPLOYED"),

    # Flags Patrimoniais (Texto -> Binário 1/0)
    F.when(F.col("FLAG_OWN_CAR") == "Y", 1).otherwise(0).cast(IntegerType()).alias("FLAG_OWN_CAR"),
    F.when(F.col("FLAG_OWN_REALTY") == "Y", 1).otherwise(0).cast(IntegerType()).alias("FLAG_OWN_REALTY"),

    # Categorias Socioeconômicas
    F.col("NAME_INCOME_TYPE").cast(StringType()),
    F.col("NAME_EDUCATION_TYPE").cast(StringType()),

    # Scores Externos (Nulos preservados para imputação no pipeline de ML)
    F.col("EXT_SOURCE_2").cast(DoubleType()),
    F.col("EXT_SOURCE_3").cast(DoubleType())
)

In [30]:
# Transformação e Higienização: bureau
df_bureau_silver = df_bureau_bronze.select(
    # Chaves Relacionais
    F.col("SK_ID_CURR").cast(IntegerType()),
    F.col("SK_ID_BUREAU").cast(IntegerType()),

    # Status e Temporalidade
    F.col("CREDIT_ACTIVE").cast(StringType()),
    F.col("DAYS_CREDIT").cast(IntegerType()),

    # Saldos e Limites (Tratamento de dívida negativa < 0 -> 0.0)
    F.col("AMT_CREDIT_SUM").cast(DoubleType()),
    F.when(F.col("AMT_CREDIT_SUM_DEBT") < 0, 0.0)
     .otherwise(F.col("AMT_CREDIT_SUM_DEBT"))
     .cast(DoubleType())
     .alias("AMT_CREDIT_SUM_DEBT")
)

## 4.2.2 Armazenamento e Validação

In [31]:
# Persistência na Camada Silver (.parquet)
app_silver_target = f"{SILVER_PATH}/application_train.parquet"
bureau_silver_target = f"{SILVER_PATH}/bureau.parquet"

df_app_silver.write \
    .mode("overwrite") \
    .parquet(app_silver_target)

df_bureau_silver.write \
    .mode("overwrite") \
    .parquet(bureau_silver_target)

print(f"application_train salvo com sucesso em: {app_silver_target}")
print(f"bureau salvo com sucesso em: {bureau_silver_target}")

[Stage 63:==============>                                          (3 + 9) / 12]

application_train salvo com sucesso em: data/silver/application_train.parquet
bureau salvo com sucesso em: data/silver/bureau.parquet


In [32]:
# Validação de Leitura dos Dados Persistidos
df_app_check = spark.read.parquet(app_silver_target)
df_bureau_check = spark.read.parquet(bureau_silver_target)

print(f"Total registros Silver (application_train): {df_app_check.count()}")
print(f"Total registros Silver (bureau): {df_bureau_check.count()}")

df_app_check.select("SK_ID_CURR", "AGE_YEARS", "YEARS_EMPLOYED", "FLAG_OWN_CAR", "EXT_SOURCE_3").show(5)
df_bureau_check.select("SK_ID_CURR", "CREDIT_ACTIVE", "AMT_CREDIT_SUM_DEBT").show(5)

Total registros Silver (application_train): 307511
Total registros Silver (bureau): 1716428
+----------+------------------+------------------+------------+------------------+
|SK_ID_CURR|         AGE_YEARS|    YEARS_EMPLOYED|FLAG_OWN_CAR|      EXT_SOURCE_3|
+----------+------------------+------------------+------------+------------------+
|    282780|45.667351129363446|4.3203285420944555|           1|              NULL|
|    282781| 46.02874743326489| 3.356605065023956|           0|0.4471785780453068|
|    282782| 40.15058179329227|1.9028062970568105|           0|              NULL|
|    282783|29.021218343600275|11.019849418206707|           0|              NULL|
|    282784|31.904175222450377|2.7214236824093088|           1|0.5673792367572691|
+----------+------------------+------------------+------------+------------------+
only showing top 5 rows
+----------+-------------+-------------------+
|SK_ID_CURR|CREDIT_ACTIVE|AMT_CREDIT_SUM_DEBT|
+----------+-------------+-----------------

---
# 5. Camada Silver $\rightarrow$  Gold: Engenharia de Features e Agregações do Bureau

## Camada Gold: Construção da ABT (Analytical Base Table)

A transição da camada **Silver** para a camada **Gold** consolida o modelo analítico no paradigma moderno de **ELT (Extract, Load, Transform)**. Enquanto a camada Silver preserva a granularidade e integridade dos dados originais por entidade de domínio (tabelas separadas e limpas), a camada Gold destina-se ao consumo analítico direto por algoritmos de Machine Learning e motores de decisão de crédito.

### O Desafio da Granularidade: Descasamento Relacional (1:1 vs. 1:N)

Para construir uma **ABT (Analytical Base Table)** voltada à classificação de risco de crédito, a unidade fundamental de análise (grão) deve ser estritamente **um registro por cliente/solicitação (`SK_ID_CURR`)**:

* **Entidade de Proposta (`application_train`)**: Grão $1:1$ (cada linha representa uma requisição única de crédito associada ao cliente e seu respectivo rótulo de inadimplência `TARGET`).
* **Entidade Externa (`bureau`)**: Grão $1:N$ (cada cliente pode ter zero, um ou dezenas de contratos de crédito passados ou vigentes no sistema financeiro geral, identificados por `SK_ID_BUREAU`).

> **Risco Arquitetural (Explosão de Granularidade por Join Ingênuo):**  
> Fazer uma junção direta entre `application_train` e `bureau` antes da sumarização geraria um efeito colateral catastrófico conhecido como **Data Duplication/Row Multiplication**. As propostas de crédito seriam duplicadas tantas vezes quantos fossem os contratos externos do cliente, inflacionando artificialmente o volume de dados, enviesando o cálculo da função de perda (Loss Function) dos modelos preditivos e vazando múltiplos registros idênticos entre treino e teste.

### Roteiro de Evidenciação Empírica

Antes de codificar qualquer junção ou cálculo analítico, executaremos as seguintes validações:
1. **Auditoria de Cardinalidade:** Distribuição e contagem de contratos no Bureau por cliente (média, mediana, máximo e dispersão).
2. **Auditoria de Ausência e Regras de Negócio em Dívidas (`AMT_CREDIT_SUM_DEBT` vs `CREDIT_ACTIVE`):** Avaliar se contratos fechados ou ativos apresentam valores nulos sistemáticos que justificam o uso de `coalesce` para zero.
3. **Auditoria de Cobertura Relacional:** Percentual de solicitantes de empréstimo presentes na base cadastral que possuem histórico no Bureau versus aqueles "desbancarizados" / sem histórico de mercado prévio.

## 5.1 Inspeção de Cardinalidade e Cobertura Relacional

In [33]:
# INSPEÇÃO DE CARDINALIDADE (DISTRIBUIÇÃO DE CONTRATOS POR CLIENTE NO BUREAU)
print("--- [1] CARDINALIDADE NO BUREAU (CONTRATOS POR CLIENTE) ---")

df_bureau_counts = (
    df_bureau_check
    .groupBy("SK_ID_CURR")
    .agg(F.count("SK_ID_BUREAU").alias("total_contratos"))
)

# Sumário estatístico da cardinalidade (média, desvio, mínimo, percentis, máximo)
df_bureau_counts.select("total_contratos").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "90%", "99%", "max"
).show()

--- [1] CARDINALIDADE NO BUREAU (CONTRATOS POR CLIENTE) ---
+-------+-----------------+
|summary|  total_contratos|
+-------+-----------------+
|  count|           305811|
|   mean|5.612708502964249|
| stddev|4.430354039722138|
|    min|                1|
|    25%|                2|
|    50%|                4|
|    75%|                8|
|    90%|               11|
|    99%|               20|
|    max|              116|
+-------+-----------------+



In [34]:
# INSPEÇÃO DE DÍVIDA (AMT_CREDIT_SUM_DEBT) vs STATUS (CREDIT_ACTIVE)
print("--- [2] COMPORTAMENTO DE NULOS EM AMT_CREDIT_SUM_DEBT POR STATUS ---")

(
    df_bureau_check
    .groupBy("CREDIT_ACTIVE")
    .agg(
        F.count("*").alias("total_registros"),
        F.sum(F.when(F.col("AMT_CREDIT_SUM_DEBT").isNull(), 1).otherwise(0)).alias("nulos_divida"),
        F.round(
            (F.sum(F.when(F.col("AMT_CREDIT_SUM_DEBT").isNull(), 1).otherwise(0)) / F.count("*")) * 100, 
            2
        ).alias("pct_nulos_divida"),
        F.sum(F.when(F.col("AMT_CREDIT_SUM_DEBT") == 0, 1).otherwise(0)).alias("qtd_divida_zero"),
        F.min("AMT_CREDIT_SUM_DEBT").alias("min_divida"),
        F.max("AMT_CREDIT_SUM_DEBT").alias("max_divida")
    )
    .orderBy(F.desc("total_registros"))
    .show(truncate=False)
)

--- [2] COMPORTAMENTO DE NULOS EM AMT_CREDIT_SUM_DEBT POR STATUS ---
+-------------+---------------+------------+----------------+---------------+----------+------------+
|CREDIT_ACTIVE|total_registros|nulos_divida|pct_nulos_divida|qtd_divida_zero|min_divida|max_divida  |
+-------------+---------------+------------+----------------+---------------+----------+------------+
|Closed       |1079273        |180543      |16.73           |892112         |0.0       |5.175E7     |
|Active       |630607         |73547       |11.66           |130064         |0.0       |1.701E8     |
|Sold         |6527           |3573        |54.74           |2665           |0.0       |3.76109019E7|
|Bad debt     |21             |6           |28.57           |11             |0.0       |182632.635  |
+-------------+---------------+------------+----------------+---------------+----------+------------+



In [35]:
# AUDITORIA DE COBERTURA RELACIONAL (APPLICATION vs BUREAU)
print("--- [3] COBERTURA RELACIONAL: APPLICATION vs BUREAU ---")

# Distinct de SK_ID_CURR nas duas tabelas
total_clientes_app = df_app_check.select("SK_ID_CURR").distinct().count()
total_clientes_bureau = df_bureau_check.select("SK_ID_CURR").distinct().count()

# Clientes presentes simultaneamente em ambas
clientes_com_historico = (
    df_app_check.select("SK_ID_CURR")
    .join(df_bureau_check.select("SK_ID_CURR").distinct(), on="SK_ID_CURR", how="inner")
    .count()
)

clientes_sem_historico = total_clientes_app - clientes_com_historico
pct_sem_historico = (clientes_sem_historico / total_clientes_app) * 100

print(f"Total de proponentes únicos em Application: {total_clientes_app}")
print(f"Total de clientes únicos registrados no Bureau: {total_clientes_bureau}")
print(f"Proponentes com histórico no Bureau (Inner Match): {clientes_com_historico} ({(clientes_com_historico / total_clientes_app) * 100:.2f}%)")
print(f"Proponentes sem histórico no Bureau: {clientes_sem_historico} ({pct_sem_historico:.2f}%)")

--- [3] COBERTURA RELACIONAL: APPLICATION vs BUREAU ---
Total de proponentes únicos em Application: 307511
Total de clientes únicos registrados no Bureau: 305811
Proponentes com histórico no Bureau (Inner Match): 263491 (85.69%)
Proponentes sem histórico no Bureau: 44020 (14.31%)


### Feature Engineering e Junção (Camada Gold)

Com base nas evidências empíricas, a construção da ABT (Analytical Base Table) aplicará agregações orientadas à análise de risco de crédito (Os 5 Cs: Character, Capacity, Capital, Collateral, Conditions).

**1. Histórico de Mercado (Character & Capacity) - Tabela Bureau:**
*   `BUREAU_TOTAL_LOANS`: Volume total de histórico do cliente (contagem de contratos).
*   `BUREAU_ACTIVE_LOANS`: Quantos empréstimos estão comprometendo a renda atualmente.
*   `BUREAU_TOTAL_DEBT`: O somatório de dívidas abertas (tratado com coalesce para 0, dado o padrão de nulos em contratos fechados).

**2. Integração (LEFT JOIN):**
*   Utilizamos `LEFT JOIN` para preservar os 14.31% de clientes "Thin File" (sem histórico de crédito prévio no mercado). As métricas de Bureau para essas pessoas serão preenchidas com `0`.

**3. Índices Financeiros Propostos (Capacity & Capital) - Tabela App:**
*   `DTI_RATIO` (Debt-to-Income): Proporção do quanto a parcela do empréstimo atual (`AMT_ANNUITY`) compromete a renda total do cliente (`AMT_INCOME_TOTAL`). Quanto maior, maior o risco.
*   `CREDIT_INCOME_RATIO`: Proporção do montante total solicitado (`AMT_CREDIT`) frente à renda anual do cliente.

## 5.2 Agregação, Join e Armazenamento da ABT

In [36]:
# FEATURE ENGINEERING: AGREGAÇÕES DO BUREAU (1:N -> 1:1)

df_bureau_agg = (
    df_bureau_check
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("SK_ID_BUREAU").alias("BUREAU_TOTAL_LOANS"),
        F.sum(F.when(F.col("CREDIT_ACTIVE") == "Active", 1).otherwise(0)).alias("BUREAU_ACTIVE_LOANS"),
        F.sum(F.coalesce(F.col("AMT_CREDIT_SUM"), F.lit(0))).alias("BUREAU_TOTAL_CREDIT_SUM"),
        F.sum(F.coalesce(F.col("AMT_CREDIT_SUM_DEBT"), F.lit(0))).alias("BUREAU_TOTAL_CREDIT_DEBT")
    )
)

In [37]:
# INTEGRAÇÃO: LEFT JOIN

# Mantém os 307.511 clientes intactos
df_gold_joined = df_app_check.join(df_bureau_agg, on="SK_ID_CURR", how="left")

# Tratamento dos 14.31% clientes sem histórico no Bureau (preenchimento com zero)
bureau_features = [
    "BUREAU_TOTAL_LOANS", "BUREAU_ACTIVE_LOANS", 
    "BUREAU_TOTAL_CREDIT_SUM", "BUREAU_TOTAL_CREDIT_DEBT"
]
df_gold_filled = df_gold_joined.fillna(0, subset=bureau_features)

In [38]:
# FEATURE ENGINEERING: ÍNDICES FINANCEIROS (APPLICATION)

df_gold_final = (
    df_gold_filled
    .withColumn("DTI_RATIO", F.round(F.col("AMT_ANNUITY") / F.col("AMT_INCOME_TOTAL"), 4))
    .withColumn("CREDIT_INCOME_RATIO", F.round(F.col("AMT_CREDIT") / F.col("AMT_INCOME_TOTAL"), 4))
)

### 5.2.1 Validação Pré Armazenamento ABT

In [39]:
df_gold_final.limit(10).toPandas()

,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,AGE_YEARS,YEARS_EMPLOYED,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,EXT_SOURCE_2,EXT_SOURCE_3,BUREAU_TOTAL_LOANS,BUREAU_ACTIVE_LOANS,BUREAU_TOTAL_CREDIT_SUM,BUREAU_TOTAL_CREDIT_DEBT,DTI_RATIO,CREDIT_INCOME_RATIO
0,100008,0,99000.0,490495.5,27517.5,454500.0,46.381930,4.347707,0,1,State servant,Secondary / secondary special,0.354225,0.621226,3,1,468445.50,240057.000,0.2780,4.9545
1,130554,1,135000.0,152820.0,15241.5,135000.0,37.366188,4.399726,0,0,Commercial associate,Secondary / secondary special,0.192432,0.528093,3,1,150102.00,26959.320,0.1129,1.1320
2,130557,0,185400.0,848745.0,43465.5,675000.0,40.566735,1.130732,0,1,Working,Higher education,0.683925,0.641368,6,4,3757077.00,1703007.000,0.2344,4.5779
3,130559,0,180000.0,331920.0,17077.5,225000.0,36.232717,16.410678,0,1,Working,Secondary / secondary special,0.441987,0.217629,3,3,968440.50,831591.000,0.0949,1.8440
4,191356,0,450000.0,1190340.0,63418.5,1125000.0,40.731006,19.531828,1,1,Commercial associate,Higher education,0.692176,0.448962,13,6,7497678.60,1755112.500,0.1409,2.6452
5,191358,0,225000.0,1190340.0,66595.5,1125000.0,52.933607,10.072553,1,1,Commercial associate,Secondary / secondary special,0.778011,0.687933,1,0,810000.00,0.000,0.2960,5.2904
6,252350,0,135000.0,405000.0,10813.5,405000.0,56.347707,8.188912,0,1,Working,Secondary / secondary special,0.299234,0.574447,6,3,1247641.29,401533.605,0.0801,3.0000
7,282788,0,157500.0,364896.0,23319.0,315000.0,41.850787,4.974675,0,0,Commercial associate,Secondary / secondary special,0.568871,0.669057,5,2,2323062.00,94779.270,0.1481,2.3168
8,282789,0,90000.0,604152.0,29196.0,540000.0,41.434634,14.937714,0,1,State servant,Secondary / secondary special,0.631605,0.391055,6,3,844510.50,427072.500,0.3244,6.7128
9,313212,0,135000.0,269550.0,14242.5,225000.0,23.548255,1.174538,1,1,Working,Secondary / secondary special,0.353785,0.661024,4,3,774108.00,375124.500,0.1055,1.9967


In [40]:
df_gold_final.printSchema()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- AGE_YEARS: double (nullable = true)
 |-- YEARS_EMPLOYED: double (nullable = true)
 |-- FLAG_OWN_CAR: integer (nullable = true)
 |-- FLAG_OWN_REALTY: integer (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- EXT_SOURCE_2: double (nullable = true)
 |-- EXT_SOURCE_3: double (nullable = true)
 |-- BUREAU_TOTAL_LOANS: long (nullable = true)
 |-- BUREAU_ACTIVE_LOANS: long (nullable = true)
 |-- BUREAU_TOTAL_CREDIT_SUM: double (nullable = false)
 |-- BUREAU_TOTAL_CREDIT_DEBT: double (nullable = false)
 |-- DTI_RATIO: double (nullable = true)
 |-- CREDIT_INCOME_RATIO: double (nullable = true)



In [43]:
df_gold_final.summary().show()

[Stage 116:>                                                        (0 + 1) / 1]

+-------+------------------+-------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+-------------------+------------------+----------------+--------------------+--------------------+--------------------+------------------+-------------------+-----------------------+------------------------+-------------------+-------------------+
|summary|        SK_ID_CURR|             TARGET| AMT_INCOME_TOTAL|       AMT_CREDIT|       AMT_ANNUITY|  AMT_GOODS_PRICE|         AGE_YEARS|    YEARS_EMPLOYED|       FLAG_OWN_CAR|   FLAG_OWN_REALTY|NAME_INCOME_TYPE| NAME_EDUCATION_TYPE|        EXT_SOURCE_2|        EXT_SOURCE_3|BUREAU_TOTAL_LOANS|BUREAU_ACTIVE_LOANS|BUREAU_TOTAL_CREDIT_SUM|BUREAU_TOTAL_CREDIT_DEBT|          DTI_RATIO|CREDIT_INCOME_RATIO|
+-------+------------------+-------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+-------------------+-------

In [ ]:
'''
-------+------------------+-------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+-------------------+------------------+----------------+--------------------+--------------------+--------------------+------------------+-------------------+-----------------------+------------------------+-------------------+-------------------+
|summary|        SK_ID_CURR|             TARGET| AMT_INCOME_TOTAL|       AMT_CREDIT|       AMT_ANNUITY|  AMT_GOODS_PRICE|         AGE_YEARS|    YEARS_EMPLOYED|       FLAG_OWN_CAR|   FLAG_OWN_REALTY|NAME_INCOME_TYPE| NAME_EDUCATION_TYPE|        EXT_SOURCE_2|        EXT_SOURCE_3|BUREAU_TOTAL_LOANS|BUREAU_ACTIVE_LOANS|BUREAU_TOTAL_CREDIT_SUM|BUREAU_TOTAL_CREDIT_DEBT|          DTI_RATIO|CREDIT_INCOME_RATIO|
+-------+------------------+-------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+-------------------+------------------+----------------+--------------------+--------------------+--------------------+------------------+-------------------+-----------------------+------------------------+-------------------+-------------------+
|  count|            307511|             307511|           307511|           307511|            307499|           307233|            307511|            252137|             307511|            307511|          307511|              307511|              306851|              246546|            307511|             307511|                 307511|                  307511|             307499|             307511|
|   mean|278180.51857657125|0.08072881945686496|168797.9192969845|599025.9997057016|27108.573909183444|538396.2074288895|43.906899566990134| 6.527499862048479|0.34010815873253314|0.6936727466659729|            NULL|                NULL|   0.514392674130847|  0.5108529061799705| 4.765114093479583|  1.762275170644302|     1675834.1430586148|       549155.4095808574|0.18093329994569038|  3.957571434192601|
| stddev|102790.17534842454| 0.2724186456483939|237123.1462788561|402490.7769958548|14493.737315118327|369446.4605400572|11.947949710569638| 6.402081210238658| 0.4737460595519281|0.4609680663599128|            NULL|                NULL|  0.1910601549849366| 0.19484436446374875| 4.496198948568981|  1.804891299543026|      3858106.108171011|      1528936.7374137868|0.09457450059399976| 2.6897282968904435|
|    min|            100002|                  0|          25650.0|          45000.0|            1615.5|          40500.0|20.503764544832308|               0.0|                  0|                 0|     Businessman|     Academic degree|8.173616518884397E-8|5.272652387098817E-4|                 0|                  0|                    0.0|                     0.0|             2.0E-4|             0.0048|
|    25%|            189134|                  0|         112500.0|         270000.0|           16524.0|         238500.0| 33.98220396988364| 2.099931553730322|                  0|                 0|            NULL|                NULL|  0.3924365604220856|  0.3706496323299817|                 1|                  0|               153823.5|                     0.0|             0.1148|             2.0187|
|    50%|            278179|                  0|         146547.0|         513531.0|           24903.0|         450000.0| 43.11841204654346| 4.509240246406571|                  0|                 1|            NULL|                NULL|  0.5659291952716554|  0.5352762504724826|                 4|                  1|               711000.0|               87604.065|             0.1628|             3.2646|
|    75%|            367112|                  0|         202500.0|         808650.0|           34596.0|         679500.0|53.880903490759756| 8.692676249144421|                  1|                 1|            NULL|                NULL|  0.6635926818710476|  0.6690566947824041|                 7|                  3|             1970446.23|                534924.0|              0.229|             5.1595|
|    max|            456255|                  1|           1.17E8|        4050000.0|          258025.5|        4050000.0| 69.07323750855578|49.040383299110196|                  1|                 1|         Working|Secondary / secon...|  0.8549996664047012|  0.8960095494948396|               116|                 32|       1.017957917385E9|    3.3449833120500004E8|              1.876|            84.7368|
+-------+------------------+-------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+-------------------+------------------+----------------+--------------------+--------------------+--------------------+------------------+-------------------+-----------------------+------------------------+-------------------+-------------------+
'''

In [47]:
# Quantidade de registros, esperado: 307511
df_gold_final.count()

307511

## 5.2.2 Armazenamento da ABT na camada Gold 

In [49]:
path_gold = "../data/gold/abt_credit_risk.parquet"
df_gold_final.write.mode("overwrite").parquet(path_gold)

print(f"Camada Gold gerada com sucesso e salva em: {path_gold}")
print(f"Dimensões finais da ABT: {df_gold_final.count()} linhas e {len(df_gold_final.columns)} variáveis (features).")

Camada Gold gerada com sucesso e salva em: ../data/gold/abt_credit_risk.parquet
Dimensões finais da ABT: 307511 linhas e 20 variáveis (features).


---
# 6 . Relatório Técnico de Conclusão da Esteira ELT (Arquitetura Medalhão)

A esteira de Engenharia de Dados executada localmente via PySpark concluiu com sucesso a ingestão, higienização, agregação analítica e persistência dos dados no formato colunar `.parquet`. O pipeline consolidou os dados das tabelas transacionais e externas na **Analytical Base Table (ABT)** em `data/gold/abt_credit_risk.parquet`.


### 1. Auditoria e Governança entre Camadas

| Camada | Objeto / Arquivo | Granularidade | Registros | Papel de Engenharia / Regras Aplicadas |
| :--- | :--- | :--- | :--- | :--- |
| **Bronze** | `application_train.csv`<br>`bureau.csv` | Transacional bruta | 307.511<br>1.716.428 | Ingestão fiel (*raw load*) sem alterações semânticas ou de valores. |
| **Silver** | `application_train.parquet`<br>`bureau.parquet` | Registro higienizado | 307.511<br>1.716.428 | Coerção de tipos; conversão de datas retroativas para anos positivos (`AGE_YEARS`); remoção do sentinela de emprego (`365243` $\rightarrow$ `Null`); truncamento de dívidas negativas (`< 0` $\rightarrow$ `0.0`); conversão de flags texto para inteiros binários (`1`/`0`). Nulos legítimos foram estritamente preservados. |
| **Gold** | `abt_credit_risk.parquet` | Proponente (`SK_ID_CURR`) | **307.511** | Agregação relacional 1:N do histórico de bureau; tratamento de clientes sem registro de mercado via `coalesce` determinístico; engenharia de métricas de endividamento e capacidade financeira (`DTI_RATIO`, `CREDIT_INCOME_RATIO`). |


### 2. Evidências Empíricas da ABT Final

* **Preservação de Escopo:** A contagem final permaneceu estritamente em **307.511 proponentes**, eliminando riscos de dispersão ou duplicação analítica.
* **Cobertura de Mercado:** 85,69% dos proponentes (263.491 clientes) tiveram seu histórico de bureau consolidado com sucesso, enquanto 14,31% (44.020 clientes) foram integrados com métricas contábeis zeradas (`0.0`), permitindo que os modelos diferenciem ausência de registro financeiro de inadimplência ativa.
* **Preservação Contra Data Leakage:** As variáveis contínuas com ausência legítima (`EXT_SOURCE_3` com 19,83% de nulos, `YEARS_EMPLOYED` com 18,01%, e resíduos em `EXT_SOURCE_2`, `AMT_GOODS_PRICE` e `AMT_ANNUITY`) permanecem isentas de qualquer imputação matemática na camada Gold, garantindo que o aprendizado da distribuição ocorra estritamente dentro da partição de treino (`X_train`) na modelagem.


### 3. Próximos Passos 
Com a persistência e a integridade da camada Gold validadas, a etapa de Engenharia de Dados encerra seu ciclo. Os próximos passos no fluxo do projeto compreendem:

1. **Transição de Ambiente:** Finalizar o fluxo de dados em PySpark e inicializar o ambiente voltado à análise exploratória e preparação dos dados para modelagem.
2. **Análise Exploratória dos Dados (EDA):** Inspecionar a distribuição do público-alvo, analisar o comportamento bivariado das variáveis derivadas em relação ao evento de risco e validar correlações entre os novos preditores.
3. **Estruturação do Pipeline de Pré-Processamento:** Definir as estratégias formais de codificação de variáveis categóricas, escalonamento numérico e tratamento dos dados ausentes a serem integradas ao estimador estatístico.